# ABAC Data Quality (DQ) Checks

## Purpose

This notebook performs **data availability checks** on every table and column used by the ABAC metrics.  
It answers the question: *"For each column we depend on, what percentage of rows have actual data (not null, not blank)?"*

Results are written to a centralized DQ results table: **`RA_FY_2025.abac_cde_da_by_metric`**

## How It Compares to the RA DQ Process

The Risk Assessment (RA) team performs the same type of check, but organized **per LOB/AU** (e.g., TDW Private Trust 101015).  
Their results go to `RA_FY_2025.cde_da_by_lob_segment`.  
This notebook adapts that process for ABAC, which is organized **per metric** (e.g., EBA01, TP01).

## DQ Formula (Same as RA)

```
NNNBV = count of rows where column IS NOT NULL AND column <> ''
total = count of all rows in the table
availability_pct = round(100 × NNNBV / total, 2)
```

- **NNNBV** stands for **"Not Null Not Blank Value"** — it's not a SQL keyword, just a naming convention used by the RA team.
- A result of **100.0** means every row has a non-null, non-empty value for that column.
- A result of **99.35** means 0.65% of rows have null or blank values.

## Data Sources — Why Everything Is ADIDO

In the Risk Assessment ecosystem, data comes from several source categories:

| Source | What It Means | Examples |
|--------|--------------|----------|
| **ADIDO** | Data ingested through the ADIDO pipeline into the `ra_adido_2025` Hive database on Databricks (Delta Lake) | Coupa expenses, Finance data, HR lists, TPRM data |
| **RZ** | Raw Zone — data from the Rahona data lake | Customer data, account data |
| **CZ** | Curated Zone — JDBC connection to SQL Server (e.g., `caedw.cust`) | UTR data, SAR data |
| **SRZ** | Secure Raw Zone | Sensitive data requiring extra access controls |

For ABAC, **every single table lives in `hive_metastore.ra_adido_2025`** — meaning all ABAC data  
was ingested through the ADIDO pipeline. Unlike the RA per-LOB checks (which mix ADIDO, RZ, and CZ sources),  
ABAC is entirely self-contained within ADIDO. This simplifies the DQ check because:
- No JDBC connections needed (no CZ/SRZ/RZ)
- All tables are directly queryable via Spark SQL
- A single `SOURCE = 'ADIDO'` applies to every check

## Metrics Covered

| Metric | Name | Type |
|--------|------|------|
| EBA01 | Travel, Lodging, and Entertainment | Count |
| EBA02 | Travel/Lodging/Entertainment (Public Officials) | Count |
| EBA04 | Gifts/Travel/Lodging Non-POs > $250 | Percentage |
| EBA06 | Donations | Count |
| EBA07 | Marketing Expenses | Count |
| EMP01 | Canadian Officer / PEP Presence | Yes/No |
| EMP03 | Attestations (ABAC & CoC Training) | Percentage |
| EMP05 | Contingent Workers in High-Risk Jurisdictions | Count |
| GEO02 | Business Travel to High-Risk Jurisdictions | Count |
| TP01 | Third Party High-Risk Jurisdictions | Count |
| TP02 | Third Party High Value (≥$1M) | Percentage |
| TP03 | Third Party Services (Marketing/Brokerage) | Yes/No |
| TP04 | Third Party Sole Source in High-Risk Jurisdictions | Count |
| TP05 | Third Party Government Interaction | Count |

**Excluded:** EMP04 (Investigations) and EMP07 (OBA) — these are scaffold notebooks with placeholder table names and are marked as not available.

---

In [ ]:
# Load database connection configurations.
# This sets up JDBC connection URLs and Azure AD Service Principal credentials.
# Even though ABAC only uses ADIDO (Spark SQL) tables, we run this config
# to maintain consistency with the RA notebook pattern and in case
# SNAPSHOT_CATALOGUE_NAME or other shared variables are defined here.
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
# Load all table/view name variables used across the Risk Assessment project.
# This defines hundreds of variables like SNAPSHOT_CATALOGUE_NAME, TABLE_NAME_DATA_AVA_SEG, etc.
# organized by category (CPB Objects, CBB Objects, ADIDO Tables, etc.).
# We specifically need SNAPSHOT_CATALOGUE_NAME (= 'RA_FY_2025') for our DQ results table.
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

In [ ]:
# Standard date setup — matches the RA team's pattern exactly.
# RUN_DATE is recorded with each DQ check so you can track results over time
# and compare runs from different dates.
import datetime
from pytz import timezone

def getTodayDate():
    tz = timezone('US/Eastern')
    df = datetime.datetime.now(tz)
    today_date = df.strftime('%Y-%m-%d')
    return today_date

today_date = getTodayDate()
print(f'Run Date: {today_date}')

In [ ]:
# ============================================================
# CREATE THE ABAC DQ RESULTS TABLE
# ============================================================
#
# This is the ABAC equivalent of the RA team's table:
#   RA_FY_2025.cde_da_by_lob_segment
#
# Key difference: RA organizes results by LOB_ID (e.g., 101015 = TDW Private Trust).
# ABAC organizes results by METRIC_IDS (e.g., EBA01, TP01).
#
# OUTPUT TABLE COLUMNS:
#   METRIC_IDS       - Which ABAC metrics depend on this column.
#                      Analogous to RA's CDE_NO field.
#                      Example: 'EBA01,EBA02,EBA04' means all three metrics use this column.
#   METRIC_DESC      - Human-readable description of what this data feeds.
#                      Example: 'Coupa Expense Data'
#   SOURCE           - Data source category. For ABAC this is always 'ADIDO'
#                      because all ABAC tables are ingested via the ADIDO pipeline
#                      into hive_metastore.ra_adido_2025.
#   SRC_TABLE_NAME   - Fully qualified table name being checked.
#                      Example: 'hive_metastore.ra_adido_2025.abac_request_paul_v3'
#   DATA_ELEMENT     - The column name being checked for data availability.
#                      Example: 'Account', 'CostCenter', 'EngagementNumber'
#   AVAILABILITY_PCT - The result: percentage of rows with non-null, non-blank values.
#                      100.0 = perfect data, <100.0 = some missing/blank values.
#   RUN_DATE         - When this check was executed. Enables historical tracking.
#
# ============================================================

TABLE_NAME = SNAPSHOT_CATALOGUE_NAME + '.abac_cde_da_by_metric'
print(f'DQ Results Table: {TABLE_NAME}')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    METRIC_IDS       STRING COMMENT 'Comma-separated ABAC metrics that depend on this column (like CDE_NO in RA)',
    METRIC_DESC      STRING COMMENT 'Human-readable description of the metric group or data category',
    SOURCE           STRING COMMENT 'Data source category — always ADIDO for ABAC (all tables in ra_adido_2025)',
    SRC_TABLE_NAME   STRING COMMENT 'Fully qualified source table name being checked',
    DATA_ELEMENT     STRING COMMENT 'Column name being checked for data availability',
    AVAILABILITY_PCT DOUBLE COMMENT 'Percentage of non-null non-blank values (0-100). 100 = perfect.',
    RUN_DATE         STRING COMMENT 'Date the check was executed (Eastern Time)'
)
''')
print('Table created/verified successfully.')

In [ ]:
def insertDQTable(metric_ids, metric_desc, source, src_table_name, data_element, availability_pct, run_date):
    '''
    Insert a single DQ check result into the ABAC DQ results table.
    This mirrors the RA team's insertDQTable() function from the LOB notebooks.

    Parameters:
        metric_ids      (str): Comma-separated metric IDs, e.g. 'EBA01,EBA02,EBA04'
        metric_desc     (str): Description, e.g. 'Coupa Expense Data'
        source          (str): Source category — always 'ADIDO' for ABAC
        src_table_name  (str): Full table name, e.g. 'hive_metastore.ra_adido_2025.abac_request_paul_v3'
        data_element    (str): Column name, e.g. 'Account'
        availability_pct(float): Result percentage, e.g. 99.35
        run_date        (str): Today's date, e.g. '2026-06-01'
    '''
    spark.sql(f'''
        INSERT INTO {TABLE_NAME}
        VALUES(
            '{metric_ids}',
            '{metric_desc}',
            '{source}',
            '{src_table_name}',
            '{data_element}',
            {availability_pct},
            '{run_date}'
        )
    ''')
    return True

In [ ]:
def checkColumnDQ(metric_ids, metric_desc, source, src_table_name, data_element, run_date):
    '''
    Check data availability for a single column in a single table.

    Uses the same NNNBV (Not Null Not Blank Value) formula as the RA team:

        availability_pct = round(100 * NNNBV / total, 2)

    where:
        total = count of ALL rows in the table
        present = count of rows where column IS NOT NULL (standard SQL count)
        NNNBV = count of rows where column IS NOT NULL AND column <> ''
                (stricter than 'present' — also excludes empty strings)

    The RA team computes 'present' but doesn't actually use it in the final result.
    We keep it here for consistency and potential debugging.

    The column is cast to STRING before checking because some columns may be
    numeric types where the empty-string check wouldn't apply — casting ensures
    consistent behavior across all data types.

    Parameters:
        metric_ids      (str): Which metrics use this column
        metric_desc     (str): Description of the data
        source          (str): Source category (always 'ADIDO' for ABAC)
        src_table_name  (str): Fully qualified table name
        data_element    (str): Column to check
        run_date        (str): Today's date

    Returns:
        float: The availability percentage (0-100)
    '''
    cast_col = f'cast({data_element} as STRING)'
    query = f'''
    SELECT round(100 * NNNBV / total, 2) as data_quality
    FROM (
        SELECT
            count(1) AS total,
            count({cast_col}) AS present,
            count(
                CASE WHEN {cast_col} IS NOT NULL
                     AND {cast_col} <> ''
                THEN 1 END
            ) AS NNNBV
        FROM {src_table_name}
    )
    '''
    data = spark.sql(query)
    df = data.toPandas()
    availability_pct = df['data_quality'].values[0]
    insertDQTable(metric_ids, metric_desc, source, src_table_name, data_element, availability_pct, run_date)
    return availability_pct

## DQ Check Configuration

The cell below defines **every table and column** used across all ABAC metric notebooks.  
This was compiled by reading through each metric notebook (EBA01, EBA02, ... TP05)  
and extracting all `FROM` clauses and column references.

### Why all tables are ADIDO

All ABAC metric notebooks query tables from `hive_metastore.ra_adido_2025`.  
This database contains data that was **ingested through the ADIDO (Ad-hoc Data Import/Output) pipeline** —  
a process where data files (CSV, Excel, etc.) are uploaded and registered as Delta Lake tables in Databricks.

Sources include: Coupa expense reports, Finance GL data, HR employee lists, TPRM third-party data,  
country risk ratings, travel reports, training records, and organizational mappings.

None of the ABAC metrics query external databases via JDBC (CZ/SRZ/RZ), unlike some RA per-LOB checks.

### How the configuration works

Each entry in the `abac_dq_checks` list specifies:
- **`metric_ids`** — which metrics depend on this table/column (analogous to `CDE_NO` in RA's DQ table)
- **`metric_desc`** — human-readable description
- **`source`** — always `'ADIDO'` for ABAC
- **`table`** — fully qualified table name
- **`columns`** — list of columns to check from that table

The execution loop (next section) iterates through this list and runs `checkColumnDQ()` for each column.

---

In [ ]:
# ============================================================
# ABAC DQ CHECK CONFIGURATION
# ============================================================
# Complete inventory of all tables and columns used by ABAC metrics.
#
# metric_ids: Which ABAC metrics depend on this table/column.
#             This is the ABAC equivalent of CDE_NO in the RA DQ table.
#             Example: 'EBA01,EBA02,EBA04' means these 3 metrics all use
#             the same Coupa expense tables.
#
# source: Data source category. Always 'ADIDO' for ABAC because all
#         tables are in hive_metastore.ra_adido_2025 (ingested via ADIDO).
#         Compare with RA checks where source can be ADIDO, RZ, or CZ.
# ============================================================

DB = 'hive_metastore.ra_adido_2025'

# --- Coupa expense tables ---
# Monthly expense report extracts used by EBA01 (all expenses),
# EBA02 (public official expenses), and EBA04 (non-PO expenses > $250).
# Each file covers a 1-2 month period within the assessment window.
COUPA_TABLES = [
    f'{DB}.ritm16070584_nov_dec2024_filtered',
    f'{DB}.ritm16070584_jan_feb2025_filtered',
    f'{DB}.ritm16070584_mar_april2025_filtered',
    f'{DB}.ritm16070584_may2025_filtered',
    f'{DB}.ritm16070584_june2025_filtered',
    f'{DB}.ritm16070584_jul_aug2025_filtered',
    f'{DB}.ritm16070584_sep_oct2025_filtered',
]

# --- Finance GL data tables ---
# General Ledger data split across 6 files, used by EBA01 (all expenses),
# EBA06 (donations - CatCodes 292, 427), and EBA07 (marketing expenses).
FINANCE_TABLES = [
    f'{DB}.f25_finance_data_1',
    f'{DB}.f25_finance_data_2',
    f'{DB}.f25_finance_data_3',
    f'{DB}.f25_finance_data_4',
    f'{DB}.f25_finance_data_5',
    f'{DB}.f25_finance_data_6',
]

# ============================================================
# Build the full DQ check list
# ============================================================
abac_dq_checks = []

# ----------------------------------------------------------
# SHARED INFRASTRUCTURE TABLES
# These tables are used by multiple (or all) metrics as
# foundational data for AU identification and bridging.
# ----------------------------------------------------------

# Master list of all 61 Assessment Units (AUs) across TD.
# Every single ABAC metric starts by querying this table
# to get the AU universe, filtered to Canadian jurisdictions.
abac_dq_checks.append({
    'metric_ids': 'ALL',
    'metric_desc': 'Shared - AU Master List (used by every metric)',
    'source': 'ADIDO',
    'table': f'{DB}.fy25_list_of_units',
    'columns': ['BusinessID', 'Business_Operational_Unit_Name', 'BusinessSegment',
                'Jurisdiction', 'US_OR_CANADA']
})

# Cost Center → AU mapping table. Used by CC-driven metrics
# (EBA, EMP, GEO) to bridge source data (which has Cost Centers)
# to Assessment Units. The 00_CC_Mapping_Setup notebook creates
# a temp view from this table with smart padding and dedup logic.
abac_dq_checks.append({
    'metric_ids': 'EBA01,EBA02,EBA04,EBA06,EBA07,EMP01,EMP03,EMP05,GEO02',
    'metric_desc': 'Shared - Cost Center to AU Mapping (CC-driven metrics)',
    'source': 'ADIDO',
    'table': f'{DB}.fy25_cost_center_mapping',
    'columns': ['CostCenterId', 'AssessableUnitID', 'AssessableUnitName',
                'Segment', 'AdditionalAssessableUnitIDandNameandSegment',
                'AdditionalAUID']
})

# ABAC-specific country risk ratings. Used by metrics that need
# to identify "high-risk jurisdictions" (FinalABACRating = 'HIGH').
abac_dq_checks.append({
    'metric_ids': 'EMP05,GEO02,TP01,TP04',
    'metric_desc': 'Shared - ABAC Country Risk Ratings (high-risk jurisdiction checks)',
    'source': 'ADIDO',
    'table': f'{DB}.td_country_risk_rating_abac',
    'columns': ['Jurisdiction', 'FinalABACRating']
})

# Third-party unit mapping. Used by all TP metrics to bridge
# TPRM engagement data to Assessment Units via LOB explosion
# and dual-bridge join logic.
abac_dq_checks.append({
    'metric_ids': 'TP01,TP02,TP03,TP04,TP05',
    'metric_desc': 'Shared - Third Party to AU Mapping (TP-driven metrics)',
    'source': 'ADIDO',
    'table': f'{DB}.third_party_unit_mapping',
    'columns': ['Assessable_Unit_ID', 'TPRM_Units']
})

# ----------------------------------------------------------
# EBA01, EBA02, EBA04 — Coupa Expense Tables
# Source: Coupa procurement system (expense reports)
# Columns:
#   Account        - Cost center/account identifier for AU bridging
#   PublicOfficial - 'Y'/'YES' or 'N'/'NO' flag (EBA02 filters for POs)
#   Total          - Dollar amount (EBA04 checks if > $250)
# ----------------------------------------------------------
for tbl in COUPA_TABLES:
    abac_dq_checks.append({
        'metric_ids': 'EBA01,EBA02,EBA04',
        'metric_desc': 'Coupa Expense Data (monthly extract)',
        'source': 'ADIDO',
        'table': tbl,
        'columns': ['Account', 'PublicOfficial', 'Total']
    })

# ----------------------------------------------------------
# EBA01, EBA06, EBA07 — Finance GL Data Tables
# Source: Finance General Ledger system
# Columns:
#   CostCenter - Cost center for AU bridging
#   CatCode    - Category code that determines the expense type
#                (EBA06 uses 292/427 for donations,
#                 EBA07 uses marketing-related codes)
# ----------------------------------------------------------
for tbl in FINANCE_TABLES:
    abac_dq_checks.append({
        'metric_ids': 'EBA01,EBA06,EBA07',
        'metric_desc': 'Finance GL Data (General Ledger extract)',
        'source': 'ADIDO',
        'table': tbl,
        'columns': ['CostCenter', 'CatCode']
    })

# ----------------------------------------------------------
# EMP01 — PEP (Politically Exposed Person) Presence
# Source: HR/Compliance PEP list
# Checks whether any employees flagged as PEPs are linked
# to an AU via their Cost Center.
# ----------------------------------------------------------
abac_dq_checks.append({
    'metric_ids': 'EMP01',
    'metric_desc': 'Employee PEP List (HR/Compliance)',
    'source': 'ADIDO',
    'table': f'{DB}.employee_pep_list_as_of_oct312025',
    'columns': ['Costcenter', 'Region']
})

# ----------------------------------------------------------
# EMP03 — Training Attestations
# Two separate tables:
#   1) ABAC training (filtered for ActivityName = 'Anti-Bribery & Anti-Corruption 2025')
#   2) Code of Conduct training
# The metric reports MIN(ABAC completion %, CoC completion %)
# ----------------------------------------------------------
abac_dq_checks.append({
    'metric_ids': 'EMP03',
    'metric_desc': 'ABAC Training Records (MLTF/Sanctions & ABAC course)',
    'source': 'ADIDO',
    'table': f'{DB}.fy25_mltf_sanctions_and_abac_11282025_all_employees',
    'columns': ['DeptID', 'FullName', 'ActivityName', 'CompletionStatus',
                'CompletionDateUTC', 'DueDate']
})

abac_dq_checks.append({
    'metric_ids': 'EMP03',
    'metric_desc': 'Code of Conduct Training Records',
    'source': 'ADIDO',
    'table': f'{DB}.td_code_of_conduct_and_ethics_110102024_10312025',
    'columns': ['DeptID', 'FullName', 'CompletionStatus',
                'CompletionDateUTC', 'DueDate']
})

# ----------------------------------------------------------
# EMP05 — Contingent Workers in High-Risk Jurisdictions
# Source: HR BI RAI enterprise employee list
# Filters for WorkerType LIKE '%contingent%' and joins with
# td_country_risk_rating_abac to identify high-risk countries.
# ----------------------------------------------------------
abac_dq_checks.append({
    'metric_ids': 'EMP05',
    'metric_desc': 'Enterprise Employee List (HR BI RAI)',
    'source': 'ADIDO',
    'table': f'{DB}.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025',
    'columns': ['CostCenterID', 'Workcountry', 'WorkerType']
})

# ----------------------------------------------------------
# GEO02 — Business Travel to High-Risk Jurisdictions
# Two travel data sources:
#   1) AMEX GBT (has Numberoftrips count column)
#   2) CWT (each row = 1 trip, no count column)
# Both joined with td_country_risk_rating_abac for high-risk filter.
# ----------------------------------------------------------
abac_dq_checks.append({
    'metric_ids': 'GEO02',
    'metric_desc': 'AMEX GBT Travel Data (with trip counts)',
    'source': 'ADIDO',
    'table': f'{DB}.compliance_country_destination_report_amex_gbt_aug12_oct31_25',
    'columns': ['CostCenter', 'DestinationCountry', 'Numberoftrips']
})

abac_dq_checks.append({
    'metric_ids': 'GEO02',
    'metric_desc': 'CWT Travel Data (1 row per trip)',
    'source': 'ADIDO',
    'table': f'{DB}.compliance_country_destination_report_cwt_nov24_aug11_25',
    'columns': ['CostCenter', 'DestinationCountry']
})

# ----------------------------------------------------------
# TP01-TP05 — Third Party Risk Management (TPRM) Data
# The main engagement table (abac_request_paul_v3) is used by
# ALL five TP metrics. Each metric filters on different columns:
#   TP01: location_of_tp, country_prod_serv_originates, Td_Countries_Impacted
#   TP02: Contract_Amount (>= $1M threshold)
#   TP03: KPI_Number, final_rating, primary_product_serv
#   TP04: Same location cols as TP01 + published_contrcats table
#   TP05: KPI_Number, final_rating
# ----------------------------------------------------------
abac_dq_checks.append({
    'metric_ids': 'TP01,TP02,TP03,TP04,TP05',
    'metric_desc': 'TPRM Engagement Data (all TP metrics)',
    'source': 'ADIDO',
    'table': f'{DB}.abac_request_paul_v3',
    'columns': ['EngagementNumber', 'owning_lob', 'lob_using', 'location_of_tp',
                'country_prod_serv_originates', 'Td_Countries_Impacted',
                'Contract_Amount', 'KPI_Number', 'final_rating',
                'primary_product_serv']
})

# Published contracts table — only used by TP04 (sole source analysis).
# Rows where EngagementNumber = 'SINGLE SOURCE' are filtered,
# and ThirdPartyName is used as the real engagement identifier.
abac_dq_checks.append({
    'metric_ids': 'TP04',
    'metric_desc': 'Published Contracts (sole source identification)',
    'source': 'ADIDO',
    'table': f'{DB}.published_contrcats_19_dec_2025',
    'columns': ['EngagementNumber', 'ThirdPartyName']
})

# ----------------------------------------------------------
# SUMMARY
# ----------------------------------------------------------
total_checks = sum(len(c['columns']) for c in abac_dq_checks)
total_tables = len(abac_dq_checks)
unique_tables = len(set(c['table'] for c in abac_dq_checks))
print(f'Configuration complete:')
print(f'  {total_checks} column checks')
print(f'  {total_tables} table entries ({unique_tables} unique tables)')
print(f'  14 metrics covered (EMP04, EMP07 excluded — not available)')

## Execute DQ Checks

The cell below loops through every configured table/column and:
1. Runs the NNNBV availability query against the table
2. Records the result percentage
3. Inserts the result into `RA_FY_2025.abac_cde_da_by_metric`

**Status flags in the output:**
- ✓ `OK` — 100% availability (perfect)
- ⚠ `WARN` — 90-99.99% availability (some missing data)
- ✗ `LOW` — below 90% availability (significant data quality issue)

---

In [ ]:
# ============================================================
# OPTIONAL: Clear previous run results
# ============================================================
# Uncomment the lines below if you want to delete results from
# a previous run on the same date before re-running.
# This prevents duplicate rows if you run the notebook multiple times.
#
# spark.sql(f"DELETE FROM {TABLE_NAME} WHERE RUN_DATE = '{today_date}'")
# print(f'Cleared previous results for {today_date}')

In [ ]:
# ============================================================
# RUN ALL DQ CHECKS
# ============================================================
# This is the main execution cell. It iterates through the
# abac_dq_checks configuration and runs checkColumnDQ() for
# each table/column combination.
#
# Results are both:
#   1. Printed to the notebook output (for immediate review)
#   2. Inserted into the DQ results table (for persistent storage)
#
# Errors are caught per-column so a single failure doesn't
# stop the entire run.
# ============================================================

results = []
errors = []

for check in abac_dq_checks:
    metric_ids = check['metric_ids']
    metric_desc = check['metric_desc']
    source = check['source']
    table = check['table']

    print(f'\n{"="*70}')
    print(f'Metrics: {metric_ids}')
    print(f'Desc:    {metric_desc}')
    print(f'Table:   {table}')
    print(f'{"="*70}')

    for col in check['columns']:
        try:
            pct = checkColumnDQ(metric_ids, metric_desc, source, table, col, today_date)
            # Classify result
            if pct == 100.0:
                status = 'OK'
                flag = '✓'
            elif pct >= 90.0:
                status = 'WARN'
                flag = '⚠'
            else:
                status = 'LOW'
                flag = '✗'
            print(f'  {flag} {col}: {pct}% [{status}]')
            results.append({
                'metric_ids': metric_ids,
                'table': table,
                'column': col,
                'availability_pct': pct,
                'status': status
            })
        except Exception as e:
            print(f'  ✗ {col}: ERROR — {str(e)}')
            errors.append({
                'metric_ids': metric_ids,
                'table': table,
                'column': col,
                'error': str(e)
            })

print(f'\n{"="*70}')
print(f'COMPLETE: {len(results)} checks passed, {len(errors)} errors')
print(f'{"="*70}')

## View Results

The cells below query the DQ results table to show:
1. **All results** — complete list of every check
2. **Flagged columns** — any column with availability below 100%
3. **Summary by table** — average and minimum availability per table
4. **Summary by metric** — average and minimum availability per metric group
5. **Error report** — any checks that failed to execute

---

In [ ]:
# ============================================================
# ALL RESULTS
# ============================================================
# Complete list of every DQ check from today's run.
# Each row shows one column check with its availability percentage.
# ============================================================
display(
    spark.sql(f'''
        SELECT *
        FROM {TABLE_NAME}
        WHERE RUN_DATE = '{today_date}'
        ORDER BY METRIC_IDS, SRC_TABLE_NAME, DATA_ELEMENT
    ''')
)

In [ ]:
# ============================================================
# FLAGGED COLUMNS — Availability below 100%
# ============================================================
# These are the columns that have some null or blank values.
# Review these to understand if the missing data impacts
# the metric calculations.
#
# STATUS:
#   WARN = 90-99.99% (minor gaps, may be acceptable)
#   LOW  = below 90% (significant issue, investigate further)
# ============================================================
display(
    spark.sql(f'''
        SELECT
            METRIC_IDS,
            METRIC_DESC,
            SRC_TABLE_NAME,
            DATA_ELEMENT,
            AVAILABILITY_PCT,
            CASE
                WHEN AVAILABILITY_PCT >= 90 THEN 'WARN'
                ELSE 'LOW'
            END AS STATUS
        FROM {TABLE_NAME}
        WHERE RUN_DATE = '{today_date}'
          AND AVAILABILITY_PCT < 100
        ORDER BY AVAILABILITY_PCT ASC
    ''')
)

In [ ]:
# ============================================================
# SUMMARY BY TABLE
# ============================================================
# Shows the overall data quality profile of each source table.
# Useful for identifying which tables have the most issues.
#
# Columns:
#   num_columns_checked - How many columns were checked in this table
#   avg_availability    - Average availability across all checked columns
#   min_availability    - Worst column in this table
#   perfect_cols        - Columns with 100% availability
#   imperfect_cols      - Columns with <100% availability
# ============================================================
display(
    spark.sql(f'''
        SELECT
            SRC_TABLE_NAME,
            METRIC_IDS,
            COUNT(*) AS num_columns_checked,
            ROUND(AVG(AVAILABILITY_PCT), 2) AS avg_availability,
            ROUND(MIN(AVAILABILITY_PCT), 2) AS min_availability,
            SUM(CASE WHEN AVAILABILITY_PCT = 100.0 THEN 1 ELSE 0 END) AS perfect_cols,
            SUM(CASE WHEN AVAILABILITY_PCT < 100.0 THEN 1 ELSE 0 END) AS imperfect_cols
        FROM {TABLE_NAME}
        WHERE RUN_DATE = '{today_date}'
        GROUP BY SRC_TABLE_NAME, METRIC_IDS
        ORDER BY min_availability ASC
    ''')
)

In [ ]:
# ============================================================
# SUMMARY BY METRIC
# ============================================================
# Shows the overall data quality health per metric group.
# Note: METRIC_IDS may contain multiple metrics (e.g. 'EBA01,EBA02,EBA04')
# because those metrics share the same source tables.
#
# If any metric group shows low min_availability, drill into the
# flagged columns (Cell above) to identify which specific column
# and table is the issue.
# ============================================================
display(
    spark.sql(f'''
        SELECT
            METRIC_IDS,
            COUNT(*) AS num_columns_checked,
            ROUND(AVG(AVAILABILITY_PCT), 2) AS avg_availability,
            ROUND(MIN(AVAILABILITY_PCT), 2) AS min_availability,
            SUM(CASE WHEN AVAILABILITY_PCT = 100.0 THEN 1 ELSE 0 END) AS perfect_cols,
            SUM(CASE WHEN AVAILABILITY_PCT < 100.0 THEN 1 ELSE 0 END) AS imperfect_cols
        FROM {TABLE_NAME}
        WHERE RUN_DATE = '{today_date}'
        GROUP BY METRIC_IDS
        ORDER BY min_availability ASC
    ''')
)

In [ ]:
# ============================================================
# ERROR REPORT
# ============================================================
# Lists any column checks that failed to execute.
# Common reasons:
#   - Column name doesn't exist in the table (typo or schema change)
#   - Table doesn't exist or was renamed
#   - Permission issues
#
# If a check errors, it means no result was written to the DQ table
# for that column. Investigate and fix before re-running.
# ============================================================
if errors:
    print(f'⚠ {len(errors)} ERRORS encountered:\n')
    for e in errors:
        print(f'  Metric:  {e["metric_ids"]}')
        print(f'  Table:   {e["table"]}')
        print(f'  Column:  {e["column"]}')
        print(f'  Error:   {e["error"]}')
        print()
else:
    print('✓ No errors — all DQ checks completed successfully.')